# 329. Longest Increasing Path in a Matrix

## Topic Alignment
- This problem models dependency resolution in DAGs (Directed Acyclic Graphs), critical for ML pipeline orchestration, build systems, and task scheduling.
- DFS with memoization on grids appears in path finding for robotics, image processing (longest monotonic paths), and terrain analysis.
- Understanding topological ordering implicit in increasing paths is fundamental for dependency management in distributed systems.

## Metadata 摘要
- Source: https://leetcode.com/problems/longest-increasing-path-in-a-matrix/
- Tags: Dynamic Programming, DFS, Memoization, Graph, Topological Sort, Matrix
- Difficulty: Hard
- Priority: High

## Problem Statement 原题描述
Given an `m x n` integers `matrix`, return the length of the longest increasing path in `matrix`.

From each cell, you can either move in four directions: left, right, up, or down. You **may not** move **diagonally** or move **outside the boundary** (i.e., wrap-around is not allowed).

**Constraints**:
- `m == matrix.length`
- `n == matrix[i].length`
- `1 <= m, n <= 200`
- `0 <= matrix[i][j] <= 2^31 - 1`

## Progressive Hints
- Hint 1: For each cell, the longest path starting from it depends on its neighbors.
- Hint 2: Use DFS from each cell to explore all possible increasing paths.
- Hint 3: Memoize results: `dp[i][j]` = longest path starting from cell (i,j).
- Hint 4: Can only move to neighbor if neighbor value is strictly greater.
- Hint 5: The increasing constraint creates an implicit DAG (no cycles possible).
- Hint 6: For each cell, path length = 1 + max(path lengths of valid neighbors).
- Hint 7: Try DFS from every cell, memoization avoids recomputation.
- Hint 8: Alternative: topological sort with in-degree counting (BFS approach).

## Solution Overview
Use **DFS with memoization**.

**Key Insight**: The strictly increasing constraint prevents cycles → forms DAG.

**State Definition**:
- `dp[i][j]`: longest increasing path starting from cell (i, j)

**Recurrence**:
```python
def dfs(i, j):
    if dp[i][j] != 0:  # Memoized
        return dp[i][j]
    
    max_len = 1  # At least the cell itself
    for di, dj in [(0,1), (0,-1), (1,0), (-1,0)]:
        ni, nj = i + di, j + dj
        if valid(ni, nj) and matrix[ni][nj] > matrix[i][j]:
            max_len = max(max_len, 1 + dfs(ni, nj))
    
    dp[i][j] = max_len
    return max_len
```

**Algorithm**: Try DFS from every cell, return maximum.

**Alternative**: Topological sort (BFS) - process cells in increasing value order.

## Detailed Explanation

### Why This Is a DAG

**Directed graph**: Edge from cell A to cell B if B is adjacent and B > A

**Acyclic**: Cannot have cycle because values strictly increase
- If A → B → C → ... → A is a cycle
- Then A < B < C < ... < A (contradiction!)

**Implication**: No need to track visited set (no infinite loops possible)

---

### DFS with Memoization

**Why memoization?**
Multiple cells may reach the same cell via different paths.

Example:
```
1 2 3
8 9 4
7 6 5
```

Cell (1,1) value 9:
- Can be reached from (1,0) value 8
- Can be reached from (0,1) value 2
- Both need longest path from (1,1)
- Compute once, reuse

---

### DFS Implementation Details

**Base case**: Current cell itself (length 1)

**Recursive case**: Try all 4 neighbors
- Only move to neighbor if neighbor > current
- Take maximum of all valid paths

**Memoization**: Store result in dp[i][j]
- Initialize to 0 (not computed)
- After computing, set to actual value
- On subsequent calls, return cached value

---

### Example Walkthrough

**Matrix**:
```
9 9 4
6 6 8
2 1 1
```

**DFS from (0,0) value 9**:
- All neighbors (9, 6, no right/down) are ≤ 9
- No valid moves
- dp[0][0] = 1

**DFS from (0,2) value 4**:
- Neighbor (0,1) = 9 > 4 ✓
- Neighbor (1,2) = 8 > 4 ✓
- Check dfs(0,1): returns 1 (computed above as 9 has no moves)
- Check dfs(1,2): 
  - Neighbor (0,2) = 4 < 8 ✗
  - Neighbor (1,1) = 6 < 8 ✗
  - Neighbor (2,2) = 1 < 8 ✗
  - dp[1][2] = 1
- dp[0][2] = 1 + max(1, 1) = 2

**DFS from (2,0) value 2**:
- Neighbor (1,0) = 6 > 2 ✓
- dfs(1,0):
  - Neighbor (0,0) = 9 > 6 ✓
  - Neighbor (1,1) = 6 = 6 ✗
  - Neighbor (2,0) = 2 < 6 ✗
  - dfs(0,0) = 1 (memoized)
  - dp[1][0] = 1 + 1 = 2
- dp[2][0] = 1 + 2 = 3

**Continue for all cells...**

Optimal path: 1 → 2 → 6 → 9 (length 4)

**Answer**: 4

---

### Why No Visited Set?

**In normal DFS**: Need visited set to avoid cycles

**Here**: Strictly increasing values prevent cycles
- Can't return to a cell with same or lower value
- Recursion naturally terminates

**Advantage**: Simpler code, less memory

---

### Alternative: Topological Sort (BFS)

**Idea**: Process cells in increasing value order

**Algorithm**:
1. Compute in-degree for each cell (number of neighbors with smaller value)
2. Start BFS from cells with in-degree 0 (local minima)
3. Process in layers, update path lengths

```python
# Initialize all cells to path length 1
dp = [[1] * n for _ in range(m)]

# Compute in-degrees
in_degree = [[0] * n for _ in range(m)]
for each cell:
    count neighbors with value < current

# BFS from cells with in-degree 0
queue = [cells with in_degree 0]
while queue:
    i, j = queue.pop()
    for each neighbor with value > current:
        dp[ni][nj] = max(dp[ni][nj], dp[i][j] + 1)
        in_degree[ni][nj] -= 1
        if in_degree[ni][nj] == 0:
            queue.append((ni, nj))
```

**Same complexity**, more complex implementation.

---

### Time Complexity Analysis

**DFS from every cell**: O(m × n) calls

**Each DFS**:
- Without memo: could visit O(m × n) cells (exponential)
- With memo: each cell computed once, O(1) lookup

**Total**: O(m × n)
- Each cell's DFS result computed exactly once
- Memoization ensures O(1) for subsequent calls
- 4 directions per cell: constant factor

**Memoization magic**: Transforms exponential to linear!

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| DFS + memo | O(m×n) | O(m×n) | Optimal, clean code |
| Topological sort | O(m×n) | O(m×n) | Same complexity, more complex |
| Pure DFS | O(4^(m×n)) | O(m×n) | Exponential without memo |
| Brute force | O((m×n)!) | O(m×n) | Try all paths, impractical |

In [ ]:
from typing import List

class Solution:
    def longestIncreasingPath(self, matrix: List[List[int]]) -> int:
        """
        DFS with memoization to find longest increasing path.
        
        Time: O(m × n)
        Space: O(m × n) for memoization
        """
        if not matrix or not matrix[0]:
            return 0
        
        m, n = len(matrix), len(matrix[0])
        dp = [[0] * n for _ in range(m)]  # Memoization table
        
        def dfs(i, j):
            """
            Returns longest increasing path starting from (i, j).
            """
            # Return memoized result if available
            if dp[i][j] != 0:
                return dp[i][j]
            
            # Base case: at least the cell itself
            max_len = 1
            
            # Try all 4 directions
            for di, dj in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                ni, nj = i + di, j + dj
                
                # Check bounds and increasing constraint
                if (0 <= ni < m and 0 <= nj < n and 
                    matrix[ni][nj] > matrix[i][j]):
                    # Recurse and take maximum
                    max_len = max(max_len, 1 + dfs(ni, nj))
            
            # Memoize and return
            dp[i][j] = max_len
            return max_len
        
        # Try DFS from every cell, return maximum
        result = 0
        for i in range(m):
            for j in range(n):
                result = max(result, dfs(i, j))
        
        return result

In [ ]:
# Test cases
tests = [
    ([[9,9,4],[6,6,8],[2,1,1]], 4),  # Path: 1→2→6→9
    ([[3,4,5],[3,2,6],[2,2,1]], 4),  # Path: 3→4→5→6
    ([[1]], 1),                       # Single cell
    ([[1,2],[4,3]], 3),              # Path: 1→2→3 or 1→4→?
]

solver = Solution()
for matrix, expected in tests:
    result = solver.longestIncreasingPath(matrix)
    assert result == expected, f"Failed for {matrix}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(m × n) where m, n are matrix dimensions
  - Each cell's longest path computed exactly once
  - Memoization ensures O(1) lookup for subsequent calls
  - 4 directions checked per cell: constant factor
- **Space**: O(m × n)
  - DP memoization table: m × n
  - Recursion stack: O(m × n) worst case (entire matrix is path)
  - Total: O(m × n)

## Edge Cases & Pitfalls
- **Single cell**: Return 1
- **All same values**: Every cell has path length 1, return 1
- **Strictly increasing row/column**: Path length equals dimension
- **Spiral pattern**: Can have very long path
- **Empty matrix**: Return 0
- **Common mistake**: Forgetting to initialize dp (using None or -1 instead of 0)
- **Common mistake**: Adding visited set (unnecessary due to DAG property)
- **Common mistake**: Allowing equal values (problem says strictly increasing)
- **Off-by-one**: Ensure boundary checks are correct (0 <= i < m, not i <= m)

## Follow-up Variants
- **Allow decreasing**: Longest monotonic path (increasing OR decreasing)
- **k-increasing**: Values must increase by at least k
- **Weighted paths**: Each cell has weight, maximize total weight
- **Limited moves**: At most k direction changes allowed
- **3D matrix**: Extend to 3D grid with 6 directions
- **Diagonal moves**: Allow 8 directions instead of 4
- **Count paths**: How many longest paths exist?
- **Online updates**: Matrix values change, recompute efficiently

## Takeaways
- **DAG property**: Strictly increasing values prevent cycles, enabling simpler DFS.
- **Memoization transforms exponential to linear** for problems with overlapping subproblems.
- **No visited set needed** when problem structure guarantees no cycles.
- DFS with memo is often cleaner than topological sort for this problem type.
- Understanding when a problem forms a DAG is crucial for algorithm design.
- This pattern extends to dependency resolution, task scheduling, and path finding.
- **Top-down DP** (memoized recursion) vs **bottom-up DP** (tabulation) - both work here.
- The increasing constraint is the key insight that simplifies the problem.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 62 | Unique Paths | DP on grid |
| LC 64 | Minimum Path Sum | DP on grid |
| LC 1091 | Shortest Path in Binary Matrix | BFS on grid |
| LC 2328 | Number of Increasing Paths | Similar with counting |
| LC 1706 | Where Will the Ball Fall | Grid DFS |
| LC 417 | Pacific Atlantic Water Flow | DFS from boundaries |